In [1]:
import os
from pathlib import Path

# Print the current working directory
starting_path = os.getcwd()

# Change directory to sleap-roots
if os.path.basename(os.getcwd()) == "sleap-roots":
    pass
else:
    os.chdir("..")
    
current_path = os.getcwd()

print(f"Starting directory: {starting_path}")
print(f"Changed to directory: {current_path}")

Starting directory: c:\Projects\sleap-roots\notebooks
Changed to directory: c:\Projects\sleap-roots


In [147]:
import numpy as np
import pandas as pd
import sleap_roots as sr
from sleap_roots.trait_pipelines import (Pipeline, DicotPipeline, YoungerMonocotPipeline, OlderMonocotPipeline, MultipleDicotPipeline)
from sleap_roots.series import Series
from sleap_roots.summary import get_summary

---

### DicotPipeline

- Examples
    - `tests/data/canola_7do`
    - `tests/data/soy_6do`

In [6]:
canola_data_folder = "tests/data/canola_7do"
canola_h5_paths = sr.find_all_h5_paths(canola_data_folder)
canola_slp_paths = sr.find_all_slp_paths(canola_data_folder)

canola_slp_paths

['tests/data/canola_7do/919QDUH.lateral.predictions.slp',
 'tests/data/canola_7do/919QDUH.primary.predictions.slp']

In [30]:
canola_data_folder = "tests/data/canola_7do"
canola_h5_paths = sr.find_all_h5_paths(canola_data_folder)
canola_slp_paths = sr.find_all_slp_paths(canola_data_folder)
canola_primary_slp = 'tests/data/canola_7do/919QDUH.primary.predictions.slp'
canola_lateral_slp = 'tests/data/canola_7do/919QDUH.lateral.predictions.slp'


# Load the data
canola = Series.load(
    series_name="canola",
    h5_path=canola_h5_paths[0],
    primary_path=canola_primary_slp,
    lateral_path=canola_lateral_slp,
)

num_frames_canola = len(canola)

In [20]:
pipeline = DicotPipeline()

pipeline.compute_frame_traits(canola)

TypeError: Pipeline.compute_frame_traits() missing 1 required positional argument: 'traits'

In [27]:
pipeline.traits

[TraitDef(name='primary_max_length_pts', fn=<function get_max_length_pts at 0x000001CBF9B30AE0>, input_traits=['primary_pts'], scalar=False, include_in_csv=False, kwargs={}, description='Points of the primary root with maximum length.'),
 TraitDef(name='pts_all_array', fn=<function get_all_pts_array at 0x000001CBF9B159E0>, input_traits=['primary_max_length_pts', 'lateral_pts'], scalar=False, include_in_csv=False, kwargs={}, description='Landmark points within a given frame as a flat arrayof coordinates.'),
 TraitDef(name='pts_list', fn=<function join_pts at 0x000001CBF9B15940>, input_traits=['primary_max_length_pts', 'lateral_pts'], scalar=False, include_in_csv=False, kwargs={}, description='A list of instance arrays, each having shape `(nodes, 2)`.'),
 TraitDef(name='root_widths', fn=<function get_root_widths at 0x000001CBF7D33E20>, input_traits=['primary_max_length_pts', 'lateral_pts'], scalar=False, include_in_csv=True, kwargs={'tolerance': 0.02, 'return_inds': False}, description='

No, a mixin doesn't need an initialization method (__init__) in Python, as it's primarily a class designed for code reuse and providing functionality, not for creating instances on its own

The order of the mixin matters.

Mixin Methods Are Available First:

Since Python resolves method lookups using C3 linearization (MRO), PipelineMixin methods will be checked first. This ensures that any debugging-related methods are available when DebuggedDicotPipeline is used.

Doesn't Affect Core Behavior:

DicotPipeline (which inherits from Pipeline) retains its normal behavior because it appears later in the inheritance list.

Doesn’t Override Anything:

Even though PipelineMixin does not override any methods, it still makes sense to keep it first. This makes it clear that it’s adding functionality rather than modifying existing behavior.

Best practice: Keep your mixin first in the inheritance list unless there’s a compelling reason not to. Your original order is correct. ✅

In [33]:
len(pipeline.traits)

50

In [36]:
pipeline.trait_computation_order

['primary_pts',
 'lateral_pts',
 'primary_max_length_pts',
 'lateral_count',
 'lateral_proximal_node_inds',
 'lateral_distal_node_inds',
 'lateral_lengths',
 'lateral_base_pts',
 'lateral_tip_pts',
 'pts_all_array',
 'pts_list',
 'root_widths',
 'primary_proximal_node_ind',
 'primary_distal_node_ind',
 'primary_length',
 'primary_base_pt',
 'primary_tip_pt',
 'lateral_angles_proximal',
 'lateral_angles_distal',
 'lateral_base_xs',
 'lateral_base_ys',
 'lateral_tip_xs',
 'lateral_tip_ys',
 'ellipse',
 'bounding_box',
 'convex_hull',
 'scanline_intersection_counts',
 'primary_angle_proximal',
 'primary_angle_distal',
 'base_ct_density',
 'network_length',
 'primary_base_pt_y',
 'primary_tip_pt_y',
 'primary_base_tip_dist',
 'base_length',
 'ellipse_a',
 'ellipse_b',
 'ellipse_ratio',
 'network_length_lower',
 'network_width_depth_ratio',
 'chull_perimeter',
 'chull_area',
 'chull_max_width',
 'chull_max_height',
 'chull_line_lengths',
 'scanline_last_ind',
 'scanline_first_ind',
 'base_m

In [43]:
len(pipeline.trait_map)

50

In [44]:
len(pipeline.traits)

50

In [239]:
class PipelineMixin:
        
    def trait_dict(self: Pipeline, plant: Series) -> dict[dict]:
        """
        Returns a dictionary where each key is a frame number and the value is the trait dict for that frame.
        Each key of the trait dict is a trait_name and the value is a trait_value
        The trait dict contains every trait possible (including intermediates), representing the entire DAG and more.
        Includes:
            - csv traits
            - summary traits (min, max, mean, median, std, p5, p25, p75, p95)
        """

        # self -> refers to a Pipeline object 
        # plant -> refers to a Series object

        num_frames = len(plant)

        trait_dict = {}

        for frame_idx in range(num_frames):

            initial_traits_dict = self.get_initial_frame_traits(plant, frame_idx)

            frame_traits = self.compute_frame_traits(initial_traits_dict)

            for trait_name in self.summary_traits:
                trait_summary = get_summary(frame_traits[trait_name], prefix = f"{trait_name}_")
                frame_traits.update(trait_summary)
            
            frame_traits["plant_name"] = plant.series_name
            frame_traits["frame_idx"] = frame_idx
            
            trait_dict[frame_idx] = frame_traits

        return trait_dict
    


    # pandas data frame

    # series_name, frame_idx, input_type, input_shape, trait_fn, output_type, output_shape
    def analyze_output_types(self, plant: Series):
        trait_dict = self.trait_dict(plant)
    
        overall = dict()

        for frame_idx in trait_dict:
            frame_traits = trait_dict[frame_idx]
            frame_dict = dict()  # This is where all traits for this frame will be stored

            for trait in frame_traits:
                curr_trait_type = type(frame_traits[trait]).__name__

                shape = None
                length = None

                if isinstance(frame_traits[trait], np.ndarray):
                    shape = frame_traits[trait].shape

                if isinstance(frame_traits[trait], list):
                    length = len(frame_traits[trait])

                frame_dict[trait] = {"value": frame_traits[trait],
                                     "dtype": curr_trait_type,
                                     "shape": shape,
                                     "length": length}

            overall[frame_idx] = frame_dict  # Store all traits for this frame in overall

        return overall
    
    def table_output(self, plant):

        output = self.analyze_output_types(plant)
        series_name = plant.series_name

        # Initialize an empty list to hold the rows
        data_rows = []

        # Loop through the outer dictionary (frame_idx)
        for frame_idx, traits in output.items():
            # Loop through the inner dictionary (trait_name)
            for trait_name, details in traits.items():
                dtype = details.get('dtype', 'N/A')
                shape = details.get('shape', 'N/A')
                length = details.get('length', 'N/A')

                # Append a row of data to the list
                data_rows.append({
                    'series_name': series_name,
                    'frame': frame_idx,
                    'trait_name': trait_name,
                    'dtype': dtype,
                    'shape': shape,
                    'length': length
                })

        # Convert the list of rows to a DataFrame
        df = pd.DataFrame(data_rows)

        # Display the DataFrame
        return df
            

class DicotPipelineDebug(PipelineMixin, DicotPipeline):
    pass


In [240]:
debug_pipeline = DicotPipelineDebug()
debugged_trait_dict = debug_pipeline.trait_dict(canola)
debugged_ = debug_pipeline.analyze_output_types(canola)

debugged_types = debug_pipeline.table_output(canola)

# {frame_idx: {trait_name: {dtype, shape, length}}}

In [247]:
len(debugged_trait_dict[0]["pts_list"])

6

In [242]:
debugged_types[debugged_types["dtype"] == "list"]

,series_name,frame,trait_name,dtype,shape,length
10,canola,0,pts_list,list,None,6.0
154,canola,1,pts_list,list,None,4.0
298,canola,2,pts_list,list,None,5.0
442,canola,3,pts_list,list,None,6.0
586,canola,4,pts_list,list,None,5.0
...,...,...,...,...,...,...
9658,canola,67,pts_list,list,None,4.0
9802,canola,68,pts_list,list,None,4.0
9946,canola,69,pts_list,list,None,6.0
10090,canola,70,pts_list,list,None,5.0


In [127]:
debug_pipeline = DicotPipelineDebug()
td = debug_pipeline.trait_dict(canola)

In [128]:
debug_pipeline.analyze_dag(canola, 0)

root_widths_mean
root_widths_median
root_widths_std
root_widths_p5
root_widths_p25
root_widths_p75
root_widths_p95
lateral_lengths_mean
lateral_lengths_median
lateral_lengths_std
lateral_lengths_p5
lateral_lengths_p25
lateral_lengths_p75
lateral_lengths_p95
scanline_intersection_counts_mean
scanline_intersection_counts_median
scanline_intersection_counts_std
scanline_intersection_counts_p5
scanline_intersection_counts_p25
scanline_intersection_counts_p75
scanline_intersection_counts_p95
lateral_angles_distal_mean
lateral_angles_distal_median
lateral_angles_distal_std
lateral_angles_distal_p5
lateral_angles_distal_p25
lateral_angles_distal_p75
lateral_angles_distal_p95
lateral_angles_proximal_mean
lateral_angles_proximal_median
lateral_angles_proximal_std
lateral_angles_proximal_p5
lateral_angles_proximal_p25
lateral_angles_proximal_p75
lateral_angles_proximal_p95
lateral_base_xs_mean
lateral_base_xs_median
lateral_base_xs_std
lateral_base_xs_p5
lateral_base_xs_p25
lateral_base_xs_p75
l

In [113]:
td[5]["lateral_pts"]

array([[[1088.63122559,  232.16430664],
        [1076.24267578,  249.02125549],
        [1052.82299805,  276.44998169]],

       [[1160.64941406,  296.45370483],
        [1140.72229004,  300.02682495],
        [1124.9675293 ,  304.79504395]],

       [[          nan,           nan],
        [1164.18493652,  224.83654785],
        [1200.3515625 ,  228.96685791]],

       [[          nan,           nan],
        [1168.32287598,  300.37463379],
        [1172.4942627 ,  304.26382446]]])

In [117]:
type(td[5]["primary_angle_distal"])

numpy.float64

In [89]:
pipeline.trait_computation_order

['primary_pts',
 'lateral_pts',
 'primary_max_length_pts',
 'lateral_count',
 'lateral_proximal_node_inds',
 'lateral_distal_node_inds',
 'lateral_lengths',
 'lateral_base_pts',
 'lateral_tip_pts',
 'pts_all_array',
 'pts_list',
 'root_widths',
 'primary_proximal_node_ind',
 'primary_distal_node_ind',
 'primary_length',
 'primary_base_pt',
 'primary_tip_pt',
 'lateral_angles_proximal',
 'lateral_angles_distal',
 'lateral_base_xs',
 'lateral_base_ys',
 'lateral_tip_xs',
 'lateral_tip_ys',
 'ellipse',
 'bounding_box',
 'convex_hull',
 'scanline_intersection_counts',
 'primary_angle_proximal',
 'primary_angle_distal',
 'base_ct_density',
 'network_length',
 'primary_base_pt_y',
 'primary_tip_pt_y',
 'primary_base_tip_dist',
 'base_length',
 'ellipse_a',
 'ellipse_b',
 'ellipse_ratio',
 'network_length_lower',
 'network_width_depth_ratio',
 'chull_perimeter',
 'chull_area',
 'chull_max_width',
 'chull_max_height',
 'chull_line_lengths',
 'scanline_last_ind',
 'scanline_first_ind',
 'base_m

In [90]:
debug_pipeline.trait_dict(canola)

Calculating trait: primary_max_length_pts
Trait name: primary_max_length_pts, Frame 0
Calculating trait: lateral_count
Trait name: lateral_count, Frame 0
Calculating trait: lateral_proximal_node_inds
Trait name: lateral_proximal_node_inds, Frame 0
Calculating trait: lateral_distal_node_inds
Trait name: lateral_distal_node_inds, Frame 0
Calculating trait: lateral_lengths
Trait name: lateral_lengths, Frame 0
Calculating trait: lateral_base_pts
Trait name: lateral_base_pts, Frame 0
Calculating trait: lateral_tip_pts
Trait name: lateral_tip_pts, Frame 0
Calculating trait: pts_all_array


KeyError: 'primary_max_length_pts'

In [78]:
pipeline.get_computation_order()[0]

'primary_pts'

In [69]:
init_traits

{'primary_pts': array([[[1016.78442383,  144.41915894],
         [1207.99304199,  304.11700439],
         [1208.89245605,  472.43710327],
         [1192.0501709 ,  656.82409668],
         [1160.87573242,  848.52990723],
         [1136.09692383, 1020.98138428]]]),
 'lateral_pts': array([[[1140.24816895,  212.87785339],
         [1156.17358398,  200.56602478],
         [          nan,           nan]],
 
        [[1112.55065918,  228.09667969],
         [1100.2980957 ,  244.82826233],
         [1072.66101074,  276.51275635]],
 
        [[1148.3215332 ,  228.33711243],
         [1200.88842773,  224.38265991],
         [1228.0637207 ,  228.92666626]],
 
        [[1204.5032959 ,  296.57699585],
         [1184.70300293,  300.33679199],
         [1172.21728516,  308.23007202]],
 
        [[          nan,           nan],
         [1204.88378906,  364.62561035],
         [1204.66918945,  368.51693726]]])}

In [73]:
pipeline.trait_computation_order[0] in init_traits

True

In [58]:
t = pipeline.get_initial_frame_traits(canola, 0)

In [60]:
t["primary_pts"]

array([[[1016.78442383,  144.41915894],
        [1207.99304199,  304.11700439],
        [1208.89245605,  472.43710327],
        [1192.0501709 ,  656.82409668],
        [1160.87573242,  848.52990723],
        [1136.09692383, 1020.98138428]]])

In [65]:
"primary_pts" in pipeline.trait_map

False